In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'jax'
os.environ['JAX_PLATFORMS'] = 'cpu'
import keras


# -----------------------------
# Building blocks
# -----------------------------

def encoder_block(x, num_filters, use_batch_norm=False, k=3):
    """
    Encoder block with pre-pool skip (as you suggested).
    Returns:
        x: downsampled tensor
        skip: pre-pool feature map for skip connection
    """
    x = keras.layers.Conv2D(num_filters, k, padding="same", use_bias=not use_batch_norm,
                            kernel_initializer="he_normal")(x)
    if use_batch_norm:
        x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation("relu")(x)

    x = keras.layers.Conv2D(num_filters, k, padding="same", use_bias=not use_batch_norm,
                            kernel_initializer="he_normal")(x)
    if use_batch_norm:
        x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation("relu")(x)

    skip = x  # pre-pool skip
    x = keras.layers.MaxPooling2D(pool_size=2)(x)
    return x, skip


def bottleneck_block(x, num_filters, use_batch_norm=False, k=3):
    """Two convs at the bottleneck (no pooling)."""
    x = keras.layers.Conv2D(num_filters, k, padding="same", use_bias=not use_batch_norm,
                            kernel_initializer="he_normal")(x)
    if use_batch_norm:
        x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation("relu")(x)

    x = keras.layers.Conv2D(num_filters, k, padding="same", use_bias=not use_batch_norm,
                            kernel_initializer="he_normal")(x)
    if use_batch_norm:
        x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation("relu")(x)
    return x


def decoder_block(x, skip, num_filters, use_batch_norm=False, k=3, upsample="transpose"):
    """
    Decoder block: upsample -> concat skip -> 2x conv.
    upsample: "transpose" (Conv2DTranspose) or "bilinear" (UpSampling2D + 1x1 Conv)
    """
    if upsample == "transpose":
        x = keras.layers.Conv2DTranspose(
            num_filters, kernel_size=2, strides=2, padding="same",
            kernel_initializer="he_normal", use_bias=True  # bias fine here
        )(x)
    else:
        x = keras.layers.UpSampling2D(size=2, interpolation="bilinear")(x)
        # project channels to num_filters after upsample
        x = keras.layers.Conv2D(
            num_filters, kernel_size=1, padding="same", use_bias=not use_batch_norm,
            kernel_initializer="he_normal"
        )(x)
        if use_batch_norm:
            x = keras.layers.BatchNormalization()(x)
        x = keras.layers.Activation("relu")(x)

    # Concatenate with the corresponding skip feature
    x = keras.layers.Concatenate()([x, skip])

    # Two convs
    x = keras.layers.Conv2D(num_filters, k, padding="same", use_bias=not use_batch_norm,
                            kernel_initializer="he_normal")(x)
    if use_batch_norm:
        x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation("relu")(x)

    x = keras.layers.Conv2D(num_filters, k, padding="same", use_bias=not use_batch_norm,
                            kernel_initializer="he_normal")(x)
    if use_batch_norm:
        x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation("relu")(x)

    return x


# -----------------------------
# U-Net Model
# -----------------------------

def unet_model(
    input_shape=(960, 960, 1),
    depth=4,
    initial_filter=64,
    encoder_kernel_size=3,
    decoder_kernel_size=3,
    output_channels=1,
    use_batch_norm=False,
    dropout_rate=0.0,                 # set >0 to enable dropout at bottleneck
    spatial_dropout=True,             # SpatialDropout2D vs Dropout
    upsample_mode="transpose",        # "transpose" or "bilinear"
    final_activation="sigmoid",
):
    """
    U-Net with pre-pool skips, mirrored decoder, and a single (Spatial)Dropout at bottleneck.

    Notes:
    - Make sure input H and W are divisible by 2**depth.
    - width/capacity controlled by `initial_filter`; context controlled by `depth`.
    """
    assert upsample_mode in ("transpose", "bilinear")

    inputs = keras.layers.Input(shape=input_shape)

    # Compute filter progression for encoder levels
    # e.g., depth=4, initial_filter=64 -> [64, 128, 256, 512]
    enc_filters = [initial_filter * (2 ** d) for d in range(depth)]
    bottleneck_filters = initial_filter * (2 ** depth)

    # ----- Encoder -----
    x = inputs
    skips = []
    for nf in enc_filters:
        x, skip = encoder_block(
            x, num_filters=nf, use_batch_norm=use_batch_norm, k=encoder_kernel_size
        )
        skips.append(skip)

    # ----- Bottleneck -----
    x = bottleneck_block(
        x, num_filters=bottleneck_filters, use_batch_norm=use_batch_norm, k=encoder_kernel_size
    )
    if dropout_rate and dropout_rate > 0.0:
        if spatial_dropout:
            x = keras.layers.SpatialDropout2D(rate=dropout_rate)(x)
        else:
            x = keras.layers.Dropout(rate=dropout_rate)(x)

    # ----- Decoder -----
    # Reverse iterate over encoder filters and corresponding skips
    for nf, skip in zip(enc_filters[::-1], skips[::-1]):
        x = decoder_block(
            x, skip=skip, num_filters=nf, use_batch_norm=use_batch_norm,
            k=decoder_kernel_size, upsample=upsample_mode
        )

    # ----- Output head -----
    outputs = keras.layers.Conv2D(
        filters=output_channels,
        kernel_size=1,
        padding="same",
        activation=final_activation,
        kernel_initializer="glorot_uniform",
        name="segmentation_head"
    )(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="UNet_PrePoolSkips")
    return model


Model: "UNet_PrePoolSkips"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 512, 512,  │        144 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 512, 512,  │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 512, 512,  │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 512, 512,  │      2,304 │ activation[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512, 512,  │         64 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 512, 512,  │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 256, 256,  │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 256, 256,  │      4,608 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 256, 256,  │      9,216 │ activation_2[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 128, 128,  │          0 │ activation_3[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 128, 128,  │     18,432 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_4[0][0]  

 Total params: 7,784,337 (29.69 MB)

 Trainable params: 7,778,321 (29.67 MB)

 Non-trainable params: 6,016 (23.50 KB)